<a href="https://colab.research.google.com/github/pretyjoshi/LgccPythonProject/blob/main/Python_project3_NYPDSchoolCrime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis of NYPD School Crime data (2017-2018)

**Tasks:**
1. Upload the data
2. Vizualize the data
3. Clean the data
4. Analyze the data
      *   Univariate (Reference data)
      *   Bivariate (With target data)
5. Provide finding and conclusion

In [ ]:
# Basic imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Plot settings (optional)
plt.style.use('default')
sns.set(style='whitegrid')

# Show versions (useful when sharing notebooks)
import sys
print('Python version:', sys.version)
print('Pandas version:', pd.__version__)
print('Seaborn version:', sns.__version__)

## 1. Load Dataset
NYPD schoool crime data, for year 2017-2018, was downloaded from https://opendata.cityofnewyork.us/

Data was uploaded to the github repository at https://github.com/pretyjoshi/LgccPythonProject

In [ ]:
# Read the uploaded CSV, containing data, from git repository
df = pd.read_csv("https://raw.githubusercontent.com/pretyjoshi/LgccPythonProject/refs/heads/main/2017_-_2018_Schools_NYPD_Crime_Data_Report_20251217.csv")
display(df.head(2))

## This file has information about the columns
col_namecsv = pd.read_csv("https://raw.githubusercontent.com/pretyjoshi/LgccPythonProject/refs/heads/main/NYPD_school_Crime_Data_Columns_Names.csv")
display(col_namecsv.head(24))
#display(col_namecsv)

## 2. Quick Data Overview

In this section we:
- Look at the first and last 5 rows
- Check shape (rows, columns)
- Get column names and data types
- Get high-level info and basic statistics

In [ ]:
# Peek at the data
print('First 5 rows:')
display(df.head())

print('Last 5 rows:')
display(df.tail())

In [ ]:
# Shape of dataset
print('Shape (rows, columns):', df.shape)

# Column names
print('\nColumn names:')
print(df.columns.tolist())

# Data types
print('\nData types:')
print(df.dtypes)

In [ ]:
# Info (non-null counts, dtypes)
print('\nDataFrame info:')
df.info()

# Basic numeric stats
print('\nDescriptive statistics (numeric columns):')
display(df.describe())

# Basic stats for categorical columns
print('\nDescriptive statistics (categorical columns):')
display(df.describe(include='object'))

# Memory usage
print('\nMemory usage (bytes):')
print(df.memory_usage(deep=True))

## 3. Missing Values Analysis

We check:
- Count of missing values
- Percentage of missing values
- A simple visual using a heatmap

In [ ]:
# Count of missing values
print('Missing values per column: count and percent')

missing_df = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": df.isnull().mean() * 100
})

# Sort table by count or percentage
missing_df = missing_df.sort_values("missing_count", ascending=False)
display(missing_df)


# Visual: missingness heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values Heatmap')
plt.show()


#### The highest number of missing values are in *"Building Name"*. However, this column would not be used for any descriptive analysis. Thus, it is fine to ignore it.

For this analysis, I am only interested in data which has criminal data reported.

In this dataset, criminal data is reported as **numric data from column 13 to 17 and 20 to 24.**. Thus,  I will focus on these colums.

Above anlsysis indicates  a very high number of missing values (>30%) for selected columns, *13 to 17 and 20 to 24.*

While checking further, it shows that most of those schools are closed. Thus, these schools have missing data for the selcted columns.

So, I will remove the rows for which more than 90% of crimnal data reporting is missing or NaN values.


In [ ]:
#df is our intiail dataFrame. We will create a new dataframe Filtered_df with reduced NaN
#Select required columns between 13-25
cols=df.iloc[:,list(range(12, 17)) + list(range(19, 24))]

#Find rows with more than 9 NaNs in those column
rows_to_remove=cols.isna().sum(axis=1)>9

#Inspect the rows that should be removed
display(df[rows_to_remove].head())

#Remove those rows and save the data as df_filtered
df_filtered=df[~rows_to_remove]
display(df_filtered.head())


In [ ]:
# Compare and visualize missing values
missing_df_combined = pd.DataFrame({
    "missing_count_original": df.isnull().sum(),
    "missing_count_filt": df_filtered.isnull().sum()
})

display(missing_df_combined)

# Visual: missingness heatmap
plt.figure(figsize=(6, 2))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values Heatmap for original data ')
plt.show()

# Visual: missingness heatmap
plt.figure(figsize=(6, 2))
sns.heatmap(df_filtered.isnull(), cbar=False)
plt.title('Missing Values Heatmap for FILTERED data')
plt.show()

print(" Filtered data shape", df_filtered.shape)

####After removing rows with all missing values, **we have data for 1175 schools.**

These are the schools for which NYPD crime data is mostly avaiable.

There are  still missing values. However, missingness in the resulting data is manageable. *Most of the missing data* is in the columns **DBN**, **Location Code**, and **Building Name**. These columns will not be used for analysis. Thus, we can ignore it for now.

### 4. Removing duplicate Rows
Now I will count duplicate rows, view them, and  create a de-duplicated version, if needed.

In [ ]:
# Number of duplicate rows
num_duplicates = df_filtered.duplicated().sum()
print('Number of duplicate rows:', num_duplicates)

# Show duplicate rows (if any)
if num_duplicates > 0:
    print('\nDuplicate rows:')
    display(df_filtered[df_filtered.duplicated()])
else:
    print('No duplicate rows found.')

# Create a version without duplicates (if you want to use it later)
if num_duplicates > 0:
  df_filtered_no_dupes = df_filtered.drop_duplicates().copy()
  print('Shape after removing duplicates:', df_filtered_no_dupes.shape)

### 5. Separate Numeric and Categorical Columns
In this section, we will identify:
- Numeric columns (int/float)
- Categorical columns (object/category)

This will help us to run appropriate plots and stats later.

In [ ]:
# Numeric columns
num_cols = df_filtered.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numeric columns:', num_cols)

# Categorical columns
cat_cols = df_filtered.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols)


**Geographical District Code** is defined as numerical. It **should be categorical or object**.


In [ ]:
df_filtered.loc[:,"Geographical District Code"] = (df_filtered["Geographical District Code"].astype("object") )
cat_cols = df_filtered.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols1)

num_unique = df_filtered[cat_cols].nunique()
print('Number of unique values:', num_unique)


## 6. Univariate Analysis
### 6.1 Numeric Features
For selected numerical features we will look at:
- Descriptive statistics
- Histograms
### 6.2 Categorical Features
For selected categorical columns we will look at:
- Value counts
- Bar plots for top categories

In [ ]:
# Descriptive statistics (transposed for readability)
print('Numeric feature statistics:')
display(df_filtered[num_cols].describe().T)

# Histograms for all numeric columns related to crime reporting second to last.
# first column '# School' is ommited.
df_filtered[num_cols[1:10]].hist(figsize=(12, 10), bins=30)
plt.suptitle('Histograms of Numeric Columns', y=1.02)
plt.tight_layout()
plt.show()


The data reflect that 75 % schools had no reported major crime wnile about 15% had 1 reported major crime.



In [ ]:
# Value counts for each categorical column
# Among all the categories,  'Borough', and 'Geographical District Code' (6 and 7) seems most informative.

combinedCat=pd.concat([df_filtered["Borough"], df_filtered["Geographical District Code"],df_filtered["ENGroupA"],df_filtered["RangeA"] ], axis=1)
display(combinedCat.head())

# Bar plots for top categories
for col in cat_cols[6:8]:
    plt.figure(figsize=(10, 3))
    df_filtered[col].value_counts(dropna=False).plot(kind='bar')
    plt.title(f'Top categories in {col} by counts')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



The figure above shows how the schools data is divided based on count, at borough and school district level.

Although there are 32 major school districs, the data also shows counts for special districts such as District 79 for alternative programs and others Districts for students with  disabilities etc.

Columns 'ENGroupA' and 'RangeA' seems inter-changeable, as both depends on number of people in the building where the school is located.

In [ ]:
## View the distribution as percentage
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(
    data=df_filtered,
    x="Borough",
    stat="percent",
    bins=30,
    ax=axes[0]
)
axes[0].set_title("Borough")

sns.histplot(
    data=df_filtered,
    x="ENGroupA",
    stat="percent",
    bins=30,
    ax=axes[1]

)
axes[1].set_title("ENGroupA")


plt.tight_layout()
plt.show()

Anslysis refect close to normal distribution.

## 7. Target Variable (Optional)

After data cleaing, we will noe select a **target column**
We will:
- Check target distribution
- Use it later for bivariate analysis

In [ ]:
# Set the target column name here
target_col = "Borough"

if target_col is not None and target_col in df_filtered.columns:

    print('Target column:', target_col)
    print('\nTarget value counts:')
    display(df_filtered[target_col].value_counts())
    print('\nTarget value percentages:')
    display(df[target_col].value_counts(normalize=True) * 100)

    plt.figure(figsize=(4,3))
    sns.countplot(x=target_col, data=df_filtered)
    plt.title(f'Distribution of {target_col}')
    plt.show()

    plt.figure(figsize=(4, 3))
    # Compute percentages
    percent_df = (df_filtered[target_col].value_counts(normalize=True).mul(100).reset_index())
    percent_df.columns = [target_col, "percent"]

    sns.barplot( x=target_col,
    y="percent",
    data=percent_df
    )

    plt.ylabel("Percent (%)")
    plt.title(f"Distribution of {target_col}")
    plt.tight_layout()
    plt.show()

else:
    print('No valid target column set. Set target_col to a column name to analyze target.')

Most of the schools are in King, folllowed by Queen and Brons.

## 8. Bivariate Analysis

If a target is defined:

- **Categorical vs Target:** countplots
- **Numeric vs Target:** boxplots and/or KDE plots

If no target, you can skip or adapt to your use case.

In [ ]:
##For comparing with Categorical colums
## ['ID', 'Building Code', 'DBN', 'Location Name', 'Location Code', 'Address', 'Borough', 'Geographical District Code',
##'Register', 'Building Name', 'Schools in Building', 'ENGroupA', 'RangeA', 'New Georeferenced Column']
## Among Categoical colums, only geogrophical ditrics code seems appropriate to compare.
cat_cols_ex_target=[ "Geographical District Code", "RangeA"]
for col in cat_cols_ex_target:
        plt.figure(figsize=(12, 4))
        sns.countplot(x=col, hue=target_col, data=df_filtered)
        plt.title(f'Distribution of {col} in {target_col}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

plt.figure(figsize=(12, 4))
sns.countplot(x=target_col, hue="RangeA", data=df_filtered)
plt.title(f'RangeA by {target_col}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Above results suggest that in full data, school with 250-500 populations are highest. They are followed by 500-750 and 750-1000. The trend is similar at Brough level.

In [ ]:
##For comparing with Numeric colums
# #Schools', 'Major N', 'Oth N', 'NoCrim N', 'Prop N', 'Vio N',
#'AvgOfMajor N', 'AvgOfOth N', 'AvgOfNoCrim N', 'AvgOfProp N', 'AvgOfVio N']
## Among Numeric colums, #Schools does not have any crime data. So we ignired it.
cat_cols_ex_target1= num_cols [1:3]
for col in cat_cols_ex_target1:
        plt.figure(figsize=(12, 4))
        sns.countplot(x=col, hue=target_col, data=df_filtered)
        plt.title(f'{col} by {target_col}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


The plots suggest that maximum schools reported No crime and followed a decresing trend with increasing number of reported crime.

In [ ]:
# Now we can further analyse and vizualze schools that have crimes rate, based on our criteria.

No_crime_School= df_filtered[df_filtered["NoCrim N"]==0]
display(No_crime_School.describe())
display(No_crime_School)

high_crime_school=df_filtered[df_filtered["Major N"]>1]
display(high_crime_school.describe())
display(high_crime_school)

cat_cols_ex_target1= num_cols [1:6]
for col in cat_cols_ex_target1:
        plt.figure(figsize=(12, 4))
        sns.countplot(x=col, hue=target_col, data=high_crime_school)
        plt.title(f'{col} by {target_col}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# view schools with very high Major N and Vio N
very_high_crime_school=df_filtered[(df_filtered["Major N"]>3) & (df_filtered["Vio N"]>3)]
display(very_high_crime_school)



About 30% (348) school reported no Major crime.

There are about 8.5 % (99/1175) schools that have high crime rate, Major N = 2 or higher.

Among them, 10 schools have very high crime rates; Both major and Violent crime as 4 or higher.


## 9. GroupBy & Aggregations (Quick Patterns)

Use this section to find patterns grouped by:
- Target
- Key crime data



In [ ]:
# Example 1: Mean of numeric columns by target (if available)
if target_col is not None and target_col in df_filtered.columns:
    print(f"Mean numeric values grouped by target '{target_col}':")
    display(df_filtered.groupby(target_col)[num_cols].mean())

average_df= df_filtered.groupby(target_col)[num_cols].mean()
average_df.T.plot(
    kind="bar",
    figsize=(14, 6)
)

plt.xlabel("Metric")
plt.ylabel("Average")
plt.title("Average Metrics by Borough")
plt.legend(title="Borough")
plt.tight_layout()
plt.show()

Non-Criminal, Other Crime and Violent crime has highest occurance in Bronx.
Major crimes are more prevalent in Manhattan.




## 10. Correlation Analysis (Numeric)
- Compute correlation matrix for numeric columns
- Visualize using a heatmap
- (If numeric target) check strongest correlations with target

In [ ]:
# Compute correlation matrix for numeric columns
corr = df_filtered[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# If target is numeric, show correlations with target
if target_col is not None and target_col in num_cols:
    print(f"Correlations with target '{target_col}':")
    display(corr[target_col].sort_values(ascending=False))

Above analysis suggests that, although there is not high correlation (<0.5) among diffent kind of reported crime for "ALL" schools, when the data correlation is compared between groupd of schools based on the  population size, the correlation is very high (>0.90).

Thus, places within similar population sizes , higher Major N is related to Higher Viol N.

## 11. Save Cleaned / EDA-Ready Data (Optional)

After cleaning (handling missing values, removing duplicates, etc.), you can save the dataset.

Update `df_clean` to your final cleaned DataFrame.

In [ ]:
# For now we just copy df; in real use, modify df first (cleaning) and then save
df_clean = df_filtered.copy()

output_file = 'cleaned_data_filtered.csv'
df_clean.to_csv(output_file, index=False)
print(f'Cleaned data saved to {output_file}')

## 12. Summary and Conclusion:

This data was obtained from the NYPD open data. IT had two file, first with actual data and second with description of columns.

 Only schools with reported crime data were included for the analysis. Thus, although 'Building Name' column has many missing values, but it is not important for this analysis, so it was ignored. Crime information is recorded in specific numeric columns, and many missing values in these columns were found to be due to schools that are closed. Schools with more than 90% missing crime data were removed. After cleaning, the dataset contains data for 1,175 schools with mostly complete NYPD crime records. The remaining missing values are mainly in identification or location-related columns, which are not used in the analysis, and there are no duplicate records.

Most schools have student populations between 250 and 500, followed by schools with 500–750 and 750–1000 students, and this pattern is consistent across boroughs. In terms of crime distribution, the Bronx reports the highest number of non-criminal, other, and violent crimes, while major crimes are most common in Manhattan.

Overall, different types of crime are not strongly related when considering all schools together. However, when schools are grouped by similar population size, the relationship between major and violent crime becomes very strong, indicating that larger crime numbers tend to occur together within similar-sized schools.

About 30% of schools reported no major crimes, while around 8.5% have high crime levels, and among these, 10 schools show very high levels of both major and violent crime.